In [ ]:
# Kaggle Notebook: XGBoost Hyperparameter Tuning (AgriSense)
# Petunjuk:
# 1. Upload file 'processed_tropical_features.csv' ke dalam Kaggle (baik melalui menu 'Add Data' atau langsung upload ke file explorer di panel kanan).
# 2. Jika Anda upload melalui 'Add Data', ubah variabel DATA_FILE di bawah menunjuk ke path tersebut (misal: '/kaggle/input/agrisense/processed_tropical_features.csv').
# 3. Jika Anda upload langsung ke workspace, biarkan DATA_FILE = "processed_tropical_features.csv".
# 4. Copy-Paste SELURUH KODE di bawah ini ke dalam SATU cell Kaggle dan klik "Run" (Tombol Play).
# 5. Anda bisa mengaktifkan Accelerator (GPU P100 / T4) di Kaggle untuk lebih mempercepat!

import pandas as pd
import numpy as np
import os
import shutil
import joblib
import json
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
from IPython.display import FileLink, display
warnings.filterwarnings('ignore')

# ==========================================
# KONFIGURASI PATH
# ==========================================
# UBAH INI JIKA DATA ANDA ADA DI FOLDER '/kaggle/input/...'
DATA_FILE = "processed_tropical_features.csv" 
ARTIFACTS_DIR = "agrisense_models"
AUDIT_FILE = os.path.join(ARTIFACTS_DIR, "tahap4_tuning_audit.json")

os.makedirs(ARTIFACTS_DIR, exist_ok=True)

FEATURE_COLS = [
    'suhu_udara', 'kelembapan_udara', 'tekanan_hpa', 'cahaya_lux',
    'kelembapan_tanah', 'suhu_tanah', 'ph_tanah', 'tvoc_ppb',
    'n_mg_kg', 'p_mg_kg', 'k_mg_kg',
    'epsilon', 'fapar', 'par',
    't_scalar', 'w_scalar', 'c_scalar',
    'gpp', 'reco', 'npp',
    'soc_baseline_gC_m2', 'c_biomass_acc', 'c_current', 'c_max',
    'elapsed_hours', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'is_daytime', 'is_cabai', 'has_full_data',
    'suhu_udara_lag1', 'kelembapan_tanah_lag1', 'co2_lag1',
    'suhu_udara_roll6', 'kelembapan_tanah_roll6',
    'vpd_approx'
]

TARGET_MAP = {
    'target_co2_ppm': 'co2_ppm',
    'target_nee_agrisense': 'nee_agrisense',
    'target_carbon_potential_score': 'carbon_potential_score',
    'target_kelembapan_tanah': 'kelembapan_tanah',
    'target_ph_tanah': 'ph_tanah'
}

print("="*60)
print("MEMUAT DATA...")
print("="*60)
df = pd.read_csv(DATA_FILE)

X = df[FEATURE_COLS].values
y = df[list(TARGET_MAP.keys())]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Normalisasi...")
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
joblib.dump(scaler_X, os.path.join(ARTIFACTS_DIR, "scaler_X.joblib"))

# ==========================================
# HYPERPARAMETER TUNING (KAGGLE OPTIMIZED)
# ==========================================
# Kaggle memiliki 4 core (CPU) atau GPU. Kita gunakan n_jobs=-1 agar semua core bekerja!
print("\nMemulai Tuning...")
audit_results = {}

param_distributions = {
    'n_estimators': [200, 300, 500, 800],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8, 10],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 2, 5]
}

for target_col, model_name in TARGET_MAP.items():
    print(f"\n---> TUNING TARGET: {model_name} <---")
    
    # MENGGUNAKAN GPU KAGGLE (P100 / T4)
    xgb_base = xgb.XGBRegressor(tree_method='hist', device='cuda', random_state=42)
    
    random_search = RandomizedSearchCV(
        estimator=xgb_base,
        param_distributions=param_distributions,
        n_iter=20,          
        scoring='r2',       
        cv=3,               
        verbose=1,
        random_state=42,
        n_jobs=-1           
    )
    
    random_search.fit(X_train_scaled, y_train[target_col].values)
    
    best_model = random_search.best_estimator_
    best_params = random_search.best_params_
    
    print(f"[*] Parameter Terbaik: {best_params}")
    
    y_pred = best_model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test[target_col].values, y_pred)
    r2 = r2_score(y_test[target_col].values, y_pred)
    
    print(f"[*] R2 Score (Test) : {r2:.4f}")
    print(f"[*] MAE (Test)      : {mae:.4f}")
    
    joblib.dump(best_model, os.path.join(ARTIFACTS_DIR, f"xgboost_{model_name}.joblib"))
    
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    top5_features = [FEATURE_COLS[i] for i in indices[:5]]
    top5_scores = [float(importances[i]) for i in indices[:5]]
    
    audit_results[model_name] = {
        'R2': float(r2),
        'MAE': float(mae),
        'Best_Params': best_params,
        'Top5_Features': dict(zip(top5_features, top5_scores))
    }

with open(AUDIT_FILE, 'w') as f:
    json.dump(audit_results, f, indent=4)

print("\n" + "="*60)
print("TUNING SELESAI! SEDANG MELAKUKAN ZIPPING...")

# Membuat file ZIP dari folder model
ZIP_NAME = "agrisense_models_tuned"
shutil.make_archive(ZIP_NAME, 'zip', ARTIFACTS_DIR)

print("="*60)
print("ZIP SELESAI! Silakan klik link di bawah ini untuk mengunduh:")
display(FileLink(f'{ZIP_NAME}.zip'))
print("="*60)
